# Hour 1 · Corpora and data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/1a_corpora_and_data.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F1a_corpora_and_data.ipynb)


**Goal:** see how a clay tablet becomes a *programmatically accessible corpus* — not an e-book, but a graph of objects (tablet → line → word → sign) with features.

We use the **Copenhagen Ugaritic Corpus (CUC)**, served as JSONL from HuggingFace and loaded with one function. Licence: **CC BY-NC 4.0** (educational use).

*Reading:* `docs/02-corpora-and-data.md`.

## 0. Setup


In [1]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. One tablet, two scripts
Every line carries both a Latin transliteration (`lines`) and the actual **cuneiform** (`ugaritic`), plus a reference (`refs`).

In [2]:
t = [x for x in texts if x["ktu"] == "1.3"][0]   # Baal Cycle
for ref, lat, cun in zip(t["refs"][:6], t["lines"][:6], t["ugaritic"][:6]):
    print(f'{ref:14s} {lat:28s} {cun}')

KTU 1.3 I 1    al . tġl[ . ]bd[xxxxx]       𐎀𐎍 𐎟 𐎚𐎙𐎍[ 𐎟 ]𐎁𐎄[xxxxx]
KTU 1.3 I 2    p rdmn . ʿbd . ali[yn]       𐎔 𐎗𐎄𐎎𐎐 𐎟 𐎓𐎁𐎄 𐎟 𐎀𐎍𐎛[𐎊𐎐]
KTU 1.3 I 3    bʿl . sid . zbl . bʿl        𐎁𐎓𐎍 𐎟 𐎒𐎛𐎄 𐎟 𐎇𐎁𐎍 𐎟 𐎁𐎓𐎍
KTU 1.3 I 4    arṣ . qm . yṯʿr              𐎀𐎗𐎕 𐎟 𐎖𐎎 𐎟 𐎊𐎘𐎓𐎗
KTU 1.3 I 5    w . yšlḥmnh                  𐎆 𐎟 𐎊𐎌𐎍𐎈𐎎𐎐𐎅
KTU 1.3 I 6    ybr d . ṯd . l pnwh          𐎊𐎁𐎗 𐎄 𐎟 𐎘𐎄 𐎟 𐎍 𐎔𐎐𐎆𐎅


## 2. The corpus is a graph of object types
In Text-Fabric the CUC objects are **tablet · column · line · word · sign**. Our loader keeps tablets, lines, word tokens, and signs.

In [3]:
print("tablets :", len(texts))
print("lines   :", sum(len(t["lines"]) for t in texts))
print("words   :", sum(len(t["tokens"]) for t in texts))
from data.loader import sign_counts
print("signs   :", sum(sign_counts(texts).values()))

tablets : 278
lines   : 7577
words   : 25477
signs   : 70490


## 3. Browse by genre
Genre labels are heuristic (KTU number + known tablets).

In [4]:
from data.loader import texts_by_genre
for g, items in sorted(texts_by_genre(texts).items(), key=lambda kv: -len(kv[1])):
    print(f'{g:20s} {len(items):3d}  e.g. {", ".join(x["ktu"] for x in items[:5])}')

letter               104  e.g. 2.1, 2.10, 2.100, 2.101, 2.102
literary/religious    90  e.g. 1.101, 1.104, 1.107, 1.108, 1.111
legal/economic        35  e.g. 3.1, 3.10, 3.11, 3.12, 3.13
ritual                15  e.g. 1.105, 1.106, 1.109, 1.112, 1.119
myth                  12  e.g. 1.1, 1.100, 1.114, 1.2, 1.23
divination            10  e.g. 1.103, 1.124, 1.140, 1.141, 1.142
epic                   9  e.g. 1.14, 1.15, 1.16, 1.17, 1.18
god-list               3  e.g. 1.102, 1.118, 1.47


## 4. Simple queries
A corpus lets us *ask questions* in code.

In [5]:
# find every tablet that contains a given word form
TARGET = "mlk"   # "king"
hits = [t["ktu"] for t in texts if TARGET in t["tokens"]]
print(f'{TARGET!r} appears in {len(hits)} tablets:', hits[:15], "...")

'mlk' appears in 89 tablets: ['1.1', '1.100', '1.103', '1.105', '1.106', '1.107', '1.108', '1.109', '1.111', '1.112', '1.115', '1.119', '1.126', '1.132', '1.139'] ...


In [6]:
# print every line of one tablet that contains the word
ktu = "1.4"
one = [t for t in texts if t["ktu"] == ktu][0]
for ref, line in zip(one["refs"], one["lines"]):
    if TARGET in line.replace(".", " ").split():
        print(f'{ref}: {line}')

KTU 1.4 I 5: [il . abh . i]l mlk
KTU 1.4 II 44: mlk
KTU 1.4 III 9: [xxx]xy . ilm . d mlk
KTU 1.4 IV 24: qrš . mlk . ab . šnnm
KTU 1.4 IV 38: dm . ʿṣm . hm . yd . il mlk
KTU 1.4 IV 48: [i]l . mlk . d yknnh . yṣḥ
KTU 1.4 VII 43: u mlk . u bl mlk
KTU 1.4 VIII 49: [spr . ilmlk . ṯ]ʿy . nqmd . mlk . ugrt


## 5. Discussion
A corpus is a **graph of objects and features**, queryable in code. CUC shares the Text-Fabric model with **BHSA** (Hebrew Bible) and **DSS** — the same queries port across corpora.

> **With full Text-Fabric** (`pip install text-fabric`; `use('DT-UCPH/cuc')`) you also get sign-level features: emendation, certainty, alternative readings.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [7]:
# Change the word and re-run. Try: "ilm" (gods), "ank" (I), "bʿl" (Baal), "ytn" (he gave)
MY_WORD = "ilm"
tablets = [t["ktu"] for t in texts if MY_WORD in t["tokens"]]
print(f'{MY_WORD!r} occurs in {len(tablets)} tablets:', tablets[:20])
# Hint: words are case-sensitive transliteration — ʿ ʾ š ḥ ṯ ġ are distinct letters.

'ilm' occurs in 69 tablets: ['1.1', '1.103', '1.104', '1.105', '1.106', '1.107', '1.112', '1.114', '1.117', '1.118', '1.124', '1.132', '1.133', '1.134', '1.139', '1.147', '1.15', '1.16', '1.17', '1.176']
